In [1]:
# -*- coding: utf-8 -*-
# Demo: FastAPI Middleware with Observability + Extensions A & B

import nest_asyncio
import uuid
import time
import logging
import structlog
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
import contextvars
from collections import defaultdict
from prometheus_client import Counter, Histogram
from prometheus_fastapi_instrumentator import Instrumentator

# Patch asyncio for notebook compatibility
nest_asyncio.apply()

print("✅ Dependencies loaded and asyncio patched")

# --------------------------
# App & ContextVar
# --------------------------
app = FastAPI(title='EY Payment API', version='1.0.0')

# ContextVar for Correlation ID (Extension B)
correlation_id_ctx = contextvars.ContextVar("correlation_id", default=None)

# --------------------------
# Logging Middleware with Correlation ID
# --------------------------
structlog.configure(
    processors=[
        structlog.stdlib.add_log_level,
        structlog.processors.TimeStamper(fmt='iso'),
        structlog.processors.JSONRenderer()
    ],
    wrapper_class=structlog.BoundLogger,
    logger_factory=structlog.PrintLoggerFactory(),
)
log = structlog.get_logger()

@app.middleware("http")
async def logging_middleware(request: Request, call_next):
    correlation_id = request.headers.get("X-Correlation-Id", str(uuid.uuid4()))
    correlation_id_ctx.set(correlation_id)

    start = time.perf_counter()

    log.info("request.started", path=request.url.path, method=request.method, correlation_id=correlation_id)

    response = await call_next(request)

    elapsed_ms = round((time.perf_counter() - start) * 1000, 2)
    log.info("request.completed", path=request.url.path, status=response.status_code, latency_ms=elapsed_ms, correlation_id=correlation_id)

    response.headers["X-Correlation-Id"] = correlation_id
    return response

# --------------------------
# Extension A: Sliding Window Rate Limiter
# --------------------------
RATE_LIMIT = 100  # requests
WINDOW_SIZE = 60  # seconds
request_store = defaultdict(list)

RATE_LIMIT_HITS = Counter("rate_limit_hits_total", "Total number of requests blocked by rate limiting")

@app.middleware("http")
async def rate_limit_middleware(request: Request, call_next):
    client_ip = request.client.host
    current_time = time.time()

    # Remove timestamps older than WINDOW_SIZE
    request_store[client_ip] = [ts for ts in request_store[client_ip] if current_time - ts < WINDOW_SIZE]

    if len(request_store[client_ip]) >= RATE_LIMIT:
        RATE_LIMIT_HITS.inc()
        retry_after = int(WINDOW_SIZE - (current_time - request_store[client_ip][0]))
        return JSONResponse(
            status_code=429,
            content={"error": "Too Many Requests", "message": f"Rate limit exceeded ({RATE_LIMIT} requests/min)"},
            headers={"Retry-After": str(retry_after)}
        )

    # Record current request timestamp
    request_store[client_ip].append(current_time)

    response = await call_next(request)
    return response

# --------------------------
# Prometheus Metrics
# --------------------------
PAYMENT_AMOUNT = Histogram(
    'payment_amount_gbp', 'Payment amount in GBP',
    buckets=[10, 50, 100, 500, 1000, 5000, 10000]
)

Instrumentator().instrument(app).expose(app)
print("✅ Prometheus metrics instrumented")

# --------------------------
# Extension B: Correlation ID Propagation via TestClient
# --------------------------
client = TestClient(app)

# --------------------------
# Sample Endpoints
# --------------------------
@app.get("/health/ready")
async def readiness():
    return {"status": "ready", "db": "ok", "mq": "ok"}

@app.get("/downstream")
async def downstream(request: Request):
    return {
        "message": "Downstream service reached",
        "received_correlation_id": request.headers.get("X-Correlation-Id")
    }

@app.get("/test-correlation")
def test_correlation():
    # Use TestClient to simulate outbound call
    correlation_id = correlation_id_ctx.get()
    response = client.get("/downstream", headers={"X-Correlation-Id": correlation_id})
    return response.json()

@app.post("/payments")
async def create_payment(request: Request):
    body = await request.json()
    correlation_id = correlation_id_ctx.get()
    log.info("payment.received", amount=body.get("amount"), currency=body.get("currency", "GBP"), correlation_id=correlation_id)
    PAYMENT_AMOUNT.observe(body.get("amount", 0))
    return {"payment_id": str(uuid.uuid4()), "status": "accepted", **body}

@app.get("/rate-test")
async def rate_test():
    return {"message": "Request allowed"}

✅ Dependencies loaded and asyncio patched
✅ Prometheus metrics instrumented


c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [5]:
response = client.get(
    "/test-correlation",
    headers={
        "X-Correlation-Id": "EY-DEMO-123"
    }
)

print(response.status_code)
print(response.json())

{"path": "/test-correlation", "method": "GET", "correlation_id": "EY-DEMO-123", "event": "request.started", "level": "info", "timestamp": "2026-06-09T05:38:54.912873Z"}
{"path": "/downstream", "method": "GET", "correlation_id": "EY-DEMO-123", "event": "request.started", "level": "info", "timestamp": "2026-06-09T05:38:54.931607Z"}
{"path": "/downstream", "status": 200, "latency_ms": 5.06, "correlation_id": "EY-DEMO-123", "event": "request.completed", "level": "info", "timestamp": "2026-06-09T05:38:54.936895Z"}
{"path": "/test-correlation", "status": 200, "latency_ms": 29.94, "correlation_id": "EY-DEMO-123", "event": "request.completed", "level": "info", "timestamp": "2026-06-09T05:38:54.942898Z"}
200
{'message': 'Downstream service reached', 'received_correlation_id': 'EY-DEMO-123'}
